In [7]:
%load_ext juliacall 

The juliacall extension is already loaded. To reload it, use:
  %reload_ext juliacall


In [9]:
""" Final code of Phase-1 with integrated Julia"""

from juliacall import Main as jl

jl.seval("""

function add(u::Vector{<:Number},v::Vector{<:Number})
        ulen = size(u)[1]
        vlen = size(v)[1]
        if ulen != vlen 
            throw(ArgumentError("dimension mismatch"))
        else 
            sum = Vector{promote_type(eltype(u),eltype(v))}(undef, ulen)
           @inbounds @simd for i = 1:1:ulen
                sum[i] = v[i]+u[i]
                           end
        return sum
        end
end
"""
 )

jl.seval( """

function sub(u::Vector{<:Number},v::Vector{<:Number})
        ulen = size(u)[1]
        vlen = size(v)[1]
        if ulen != vlen 
            throw(ArgumentError("dimension mismatch"))
        else 
            sum = Vector{promote_type(eltype(u),eltype(v))}(undef, ulen)
           @inbounds @simd for i = 1:1:ulen
                sum[i] = u[i]-v[i]
                           end
        return sum
        end
end
"""
 )

jl.seval("""

function scale(u::Vector{<:Number},sf::Number)
    scaled_vector = Vector{promote_type(eltype(u),eltype(sf))}(undef,size(u)[1])
    @inbounds @simd for i in 1:1:size(u)[1]
        scaled_vector[i] = sf*u[i]
    end
    return scaled_vector
end
""")
jl.seval("""
function mag(u::Vector{<:Number})
    a = 0
    @inbounds @simd for i in 1:1:size(u)[1]
        a += u[i]^2
    end
    return a^0.5
end
""")

jl.seval("""

function dot(u::Vector{<:Number},v::Vector{<:Number})
    ulen = size(u)[1]
    vlen = size(v)[1]
    if ulen != vlen
        throw(ArgumentError("Dimension Mismatch"))
    else 
        a = 0
        for i in 1:1:ulen
            a += u[i]*v[i]
        end
    return a
    end
end
""")



class vec():
    def __init__(self,*comp) : # comp stands for components
        self.__comp = jl.convert(jl.Vector[jl.Number],list(comp))
    def len(self): 
        return jl.size(self.__comp)[0]
    def get_vec(self):
        return self.__comp
    def __repr__(self):
        # Leveraging Julia's own function 
        return jl.sprint(jl.show, jl.MIME("text/plain"), self.__comp)
    def __add__(p1,p2) :
        return vec(*jl.add(p1.get_vec(),p2.get_vec()))
    def __sub__(p1,p2) :  # Gives p1-p2
        return vec(*jl.sub(p1.get_vec(),p2.get_vec())) 
    def scale(v,a = 1) :
        return vec(*jl.scale(v.get_vec(),a))
    def mag(v) :
       return jl.mag(v.get_vec())
    
    def dot(u,v) :
        return jl.dot(u.get_vec(),v.get_vec())
    def cosine(u,v):
        w=  vec.dot(u,v)/(u.mag()*v.mag())
        return   w
    def normalise(self) :
        w = self.mag()
        z = self.scale(1/w)
        return z
        
    


v1 = vec(1,2,3)
v2 = vec(4,5,6)

print(v1+v2)
print("-----------------------")
print(v1-v2)
print("-----------------------")
print(v1.mag())
print("-----------------------")
print(v1.dot(v2))
print("-----------------------")
print(v1.scale(2))
print("-----------------------")
print(v1.cosine(v2))
print("-----------------------")
print(v1.normalise())
print("-----------------------")
vec(1,2,3).dot(vec(1,2))

3-element Vector{Number}:
 5
 7
 9
-----------------------
3-element Vector{Number}:
 -3
 -3
 -3
-----------------------
3.7416573867739413
-----------------------
32
-----------------------
3-element Vector{Number}:
 2
 4
 6
-----------------------
0.9746318461970762
-----------------------
3-element Vector{Number}:
 0.2672612419124244
 0.5345224838248488
 0.8017837257372732
-----------------------


JuliaError: ArgumentError: Dimension Mismatch
Stacktrace:
 [1] dot(u::Vector{Number}, v::Vector{Number})
   @ Main .\none:5
 [2] pyjlany_call(self::typeof(dot), args_::Py, kwargs_::Py)
   @ PythonCall.JlWrap C:\Users\atharv shrivastav\.julia\packages\PythonCall\JksWe\src\JlWrap\any.jl:48
 [3] _pyjl_callmethod(f::Any, self_::Ptr{PythonCall.C.PyObject}, args_::Ptr{PythonCall.C.PyObject}, nargs::Int64)
   @ PythonCall.JlWrap C:\Users\atharv shrivastav\.julia\packages\PythonCall\JksWe\src\JlWrap\base.jl:71
 [4] _pyjl_callmethod(o::Ptr{PythonCall.C.PyObject}, args::Ptr{PythonCall.C.PyObject})
   @ PythonCall.JlWrap.Cjl C:\Users\atharv shrivastav\.julia\packages\PythonCall\JksWe\src\JlWrap\C.jl:72

In [18]:

""" Finalised version of  making a small vector object/ Only julia Integration left"""



class vec():
    def __init__(self,*comp) :
        self.__comp = list(comp)
    def len(self): 
        return len(self.__comp)
    def get_vec(self):
        return self.__comp
    def add(p1,p2) :
        if vec.len(p1) != vec.len(p2) : 
           raise TypeError("dimension mismatch")
            
        else :
            p_net = vec()
            for i in range(0,vec.len(p1),1):
                p_net.get_vec().append(p1.get_vec()[i]+p2.get_vec()[i])
            return p_net
    def add_all(v1,*arg):
        for i in range(0,len(arg),1):
            if v1.len() != arg[i].len() :
                raise TypeError("dimensionally not equal")
            else :
                for j in range(0,arg[i].len(),1):
                    v1.get_vec()[j] = v1.get_vec()[j] + arg[i].get_vec()[j]
        return v1

    def scale(v,a = 1) :
        dummyvec = vec()
        for i in range(0,v.len(),1):
            dummyvec.get_vec().append(v.get_vec()[i]*a)
        return dummyvec
    def mag(v) :
        a = 0
        for i in range(0,v.len(),1):
            a = a+ v.get_vec()[i]**2
        return a**(0.5)
    
    def dot(u,v) :
        a = 0 
        if u.len() != v.len() :
            raise TypeError("dimension mismatch")
        else :
            for i in range(0,u.len(),1):
                a = a +u.get_vec()[i]*v.get_vec()[i]
            return a
    def cosine(u,v):
        w=  vec.dot(u,v)/(u.mag()*v.mag())
        return   w
    def normalise(self) :
        w = self.mag()
        z = self.scale(1/w)
        return z
    




                    
v1 = vec(11.0,12.0,13.0,14.0,15.0)
v2 = vec(3,4)
v4 = vec(1,2,2,2,10)
v3 = vec(1,2,2)
v5 = vec(12,2,8,27,23)



c = v2.normalise() 
c.mag()



1.0

In [2]:
 """ Second attempt of making a small vector object"""



class vec():
    def __init__(self,*comp) :
        self.comp = list(comp)
    def len(self): 
        return len(self.comp)
    def add(p1,p2) :
        if vec.len(p1) != vec.len(p2) : 
            return "dimensionally not equal"
            
        else :
            p_net = vec()
            for i in range(0,vec.len(p1),1):
                p_net.comp.append(p1.comp[i]+p2.comp[i])
            return p_net
            
    def add_all(*vecas) :
          if vec.len(vecas[0]) != vec.len(vecas[1]) :
              return "dimensionally not equal"
          else : 
              dummyvector = vec.add(vecas[0],vecas[1])
          for i in range(2,len(vecas)):
              if vec.len(dummyvector) != vec.len(vecas[i]): 
                  return "dimensionally not equal"
                  break
              else: 
                  dummyvector = vec.add(dummyvector,vecas[i])
          return dummyvector.comp             
        
         
v1 = vec(2,3,4,51,9)
v2 = vec(1,2,3,4,5)
v4 = vec(1,2,2,2,10)
v3 = vec(1,2,2)
v5 = vec(12,2,8,27,23)


print(vec.add_all(v1,v2))
print(v4.comp)
print(vec.add_all(v2,v1,v4,v5))
print(vec.add_all(v2,v1,v3,v5))
print(vec.add_all(v3,v5))
v3.comp.extend([1,2])
print(vec.add_all(v3,v5))

[3, 5, 7, 55, 14]
[1, 2, 2, 2, 10]
[16, 9, 17, 84, 47]
dimensionally not equal
dimensionally not equal
[13, 4, 10, 28, 25]


In [1]:

''' Earliest attempt of making a small vector object'''

class vec():
    def __init__(self,x,y) :
        self.x = x
        self.y = y

    def distance(pt1,pt2):
        delta_x = pt1.x-pt2.x
        delta_y = pt1.y-pt2.y
        D = (delta_x**2 + delta_y**2)**0.5
        return D

    def add(pt1,pt2) :
        x_net = pt1.x+pt2.x
        y_net = pt1.y+pt2.y
        c_net = vec(x_net,y_net)
        return c_net 

    def dot(pt1,pt2) :
        x_comp  = pt1.x*pt2.x
        y_comp = pt1.y*pt2.y
        product = x_comp+y_comp
        return product
    def mag(pt1) :
        return (pt1.x**2+pt1.y**2)**0.5
    


v1 = vec(1,1)
v2 = vec(3,4)
w =  vec.add(v1,v2) ## adds two vectors
print ( vec.distance(v1,v2)) # gives distance between two position vectors
print(w.x,",",w.y) # prints the resultant of  two added  vectors
print(vec.mag(v1),",", vec.mag(v2)) #gives magnitude of vector v1 and vector v2
print(vec.dot(v1,v2)) #gives dot product of vector v1,v2
print( "cosine angle between v1 and v2: ", vec.dot(v1,v2)/(vec.mag(v1)*vec.mag(v2)))

3.605551275463989
4 , 5
1.4142135623730951 , 5.0
7
cosine angle between v1 and v2:  0.9899494936611665
